# Week 9 Lab 2: Sequence Modeling & The Functional API

> **Goal**: Transition from simple stacks (Sequential) to complex graphs (Functional API). We will build an LSTM-based sequence model to understand how temporal data flows through a network.

## 1. Why the Functional API?

Until now, we have used `models.Sequential`. This is great for simple stacks, but real-world models (like Transformers) often have:
- Multiple inputs/outputs
- Residual connections (skipping layers)
- Shared layers

The Functional API treats layers like mathematical functions: `output = layer(input)`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

# 1. Define the Input shape
inputs = layers.Input(shape=(100,))

# 2. Chain the layers like functions
x = layers.Embedding(input_dim=5000, output_dim=64)(inputs)
x = layers.LSTM(128, return_sequences=False)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

# 3. Create the Model object
model = Model(inputs=inputs, outputs=outputs)

model.summary()

## 2. Sequence Prediction Task

We will build a model that predicts the next number in a sine wave. This is the simplest form of 'Generative' AI.

In [ ]:
# Generate a sine wave
t = np.linspace(0, 100, 1000)
data = np.sin(t)

def create_sequences(data, seq_length=50):
    x = []
    y = []
    for i in range(len(data) - seq_length):
        x.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(x), np.array(y)

seq_length = 50
x_train, y_train = create_sequences(data, seq_length)
x_train = x_train.reshape(-1, seq_length, 1) # LSTM expects (batch, steps, features)

print(f"Sequence shape: {x_train.shape}")

## 3. Building the LSTM (Functional)

Now we build a deeper model using the Functional API to prepare for the Transformer blocks in Week 10.

In [ ]:
inputs = layers.Input(shape=(seq_length, 1))

# First LSTM layer (returns full sequence for the next LSTM)
x = layers.LSTM(64, return_sequences=True)(inputs)

# Second LSTM layer (returns only the final state)
x = layers.LSTM(64, return_sequences=False)(x)

outputs = layers.Dense(1)(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='mse')

model.fit(x_train, y_train, epochs=10, batch_size=32, verbose=1)

## 4. Visualizing Results
We can see how the model "hallucinates" the future of the sine wave.

In [ ]:
import matplotlib.pyplot as plt

test_seq = x_train[0:1]
preds = []

for _ in range(100):
    p = model.predict(test_seq, verbose=0)
    preds.append(p[0,0])
    # Append prediction and slide window
    new_val = p.reshape(1, 1, 1)
    test_seq = np.concatenate([test_seq[:, 1:, :], new_val], axis=1)

plt.plot(preds, label='Generated Sine Wave')
plt.legend()
plt.show()